In [1]:
!pip install gym-super-mario-bros==7.3.2 -q
!pip install nes-py -q
!pip install opencv-python -q
!pip install imageio[ffmpeg] -q
!apt-get install ffmpeg -y -q
!apt-get install -y xvfb -q
!pip install pyvirtualdisplay -q

from pyvirtualdisplay import Display
display = Display(visible=0, size=(256, 240))
display.start()
print("Virtual display started")

import nes_py._rom as _rom_module
import gym_super_mario_bros.smb_env as _smb_env
import gym.wrappers.time_limit as _time_limit
import numpy as np

def _patched_prg_rom_stop(self):
    return int(self.prg_rom_start) + int(self.prg_rom_size) * 1024

def _patched_chr_rom_stop(self):
    return int(self.chr_rom_start) + int(self.chr_rom_size) * 1024

_rom_module.ROM.prg_rom_stop = property(_patched_prg_rom_stop)
_rom_module.ROM.chr_rom_stop = property(_patched_chr_rom_stop)

def _patched_x_position(self):
    return int(self.ram[0x6d]) * 0x100 + int(self.ram[0x86])

def _patched_y_pixel(self):
    return int(self.ram[0x03b8])

def _patched_y_position(self):
    return int(self.ram[0x00b5])

_smb_env.SuperMarioBrosEnv._x_position = property(_patched_x_position)
_smb_env.SuperMarioBrosEnv._y_pixel = property(_patched_y_pixel)
_smb_env.SuperMarioBrosEnv._y_position = property(_patched_y_position)

np.bool8 = np.bool_

def _patched_time_limit_step(self, action):
    observation, reward, done, info = self.env.step(action)
    self._elapsed_steps += 1
    if self._elapsed_steps >= self._max_episode_steps:
        info["TimeLimit.truncated"] = not done
        done = True
    return observation, reward, done, info

_time_limit.TimeLimit.step = _patched_time_limit_step

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical
import gym
from gym import spaces
import gym_super_mario_bros
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT
from nes_py.wrappers import JoypadSpace
import cv2
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print("All patches applied")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.7/77.7 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.6/198.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 31.2 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 141 not upgraded.
Reading package lists...
Building dependency tree...
Reading state information...
xvfb is already the newest version (2:21.1.4-2ubuntu1.7~22.04.16).
0 upgraded, 0 newly installed, 0 to remove and 141 not upgraded.
Virtual display started


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Device: cuda
All patches applied


In [2]:
def process_frame(frame):
    if frame is not None:
        frame = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        frame = cv2.resize(frame, (84, 84))[None, :, :] / 255.
        return frame
    else:
        return np.zeros((1, 84, 84))


class CustomReward(gym.Wrapper):
    def __init__(self, env=None, world=None, stage=None):
        super(CustomReward, self).__init__(env)
        self.observation_space = spaces.Box(low=0, high=255, shape=(1, 84, 84))
        self.curr_score = 0
        self.current_x = 40
        self.world = world
        self.stage = stage

    def step(self, action):
        result = self.env.step(action)
        if len(result) == 5:
            state, reward, terminated, truncated, info = result
            done = terminated or truncated
        else:
            state, reward, done, info = result
        state = process_frame(state)
        reward += (info["score"] - self.curr_score) / 40.
        self.curr_score = info["score"]
        if done:
            if info["flag_get"]:
                reward += 50
            else:
                reward -= 50
        if self.world == 8 and self.stage == 4:
            if info["x_pos"] < self.current_x - 200:
                reward -= 50
                done = True
        self.current_x = info["x_pos"]
        return state, reward / 10., done, info

    def reset(self):
        self.curr_score = 0
        self.current_x = 40
        result = self.env.reset()
        if isinstance(result, tuple):
            state = result[0]
        else:
            state = result
        return process_frame(state)


class CustomSkipFrame(gym.Wrapper):
    def __init__(self, env, skip=4):
        super(CustomSkipFrame, self).__init__(env)
        self.observation_space = spaces.Box(low=0, high=255, shape=(skip, 84, 84))
        self.skip = skip
        self.states = np.zeros((skip, 84, 84), dtype=np.float32)

    def step(self, action):
        total_reward = 0
        last_states = []
        for i in range(self.skip):
            state, reward, done, info = self.env.step(action)
            total_reward += reward
            if i >= self.skip / 2:
                last_states.append(state)
            if done:
                self.reset()
                return self.states.astype(np.float32), total_reward, done, info
        max_state = np.max(np.concatenate(last_states, 0), 0)
        self.states[:-1] = self.states[1:]
        self.states[-1] = max_state
        return self.states.astype(np.float32), total_reward, done, info

    def reset(self):
        result = self.env.reset()
        if isinstance(result, tuple):
            state = result[0]
        else:
            state = result
        self.states = np.concatenate([state for _ in range(self.skip)], 0)
        return self.states.astype(np.float32)


class ActorCritic(nn.Module):
    def __init__(self, num_inputs, num_actions):
        super(ActorCritic, self).__init__()
        self.conv1 = nn.Conv2d(num_inputs, 32, 3, stride=2, padding=1)
        self.conv2 = nn.Conv2d(32, 32, 3, stride=2, padding=1)
        self.conv3 = nn.Conv2d(32, 32, 3, stride=2, padding=1)
        self.conv4 = nn.Conv2d(32, 32, 3, stride=2, padding=1)
        self.fc = nn.Linear(32 * 6 * 6, 512)
        self.actor = nn.Linear(512, num_actions)
        self.critic = nn.Linear(512, 1)
        self._initialize_weights()

    def _initialize_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):
                nn.init.orthogonal_(module.weight, nn.init.calculate_gain('relu'))
                nn.init.constant_(module.bias, 0)

    def forward(self, x):
        x = x.float() / 255.0
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = F.relu(self.fc(x.view(x.size(0), -1)))
        return x

    def get_action(self, x):
        x = self.forward(x)
        policy = F.softmax(self.actor(x), dim=-1)
        dist = Categorical(policy)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        return action.item(), log_prob


def make_env(world, stage):
    env = gym_super_mario_bros.make(f"SuperMarioBros-{world}-{stage}-v0")
    env = JoypadSpace(env, SIMPLE_MOVEMENT)
    env = CustomReward(env, world, stage)
    env = CustomSkipFrame(env)
    return env


num_actions = len(SIMPLE_MOVEMENT)
model = ActorCritic(4, num_actions).to(device)
checkpoint = torch.load("/kaggle/input/models/joercharles/mariobot1-1/pytorch/default/1/mario_ppo.pt", map_location=device)
model.load_state_dict(checkpoint["model"])
model.eval()
print(f"Model loaded from update {checkpoint['update']}")
print("Ready to record!")

Model loaded from update 5000
Ready to record!


In [3]:
import imageio
import numpy as np
import torch
from PIL import Image
import gym_super_mario_bros
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT
from nes_py.wrappers import JoypadSpace


def record_agent(model, world, stage, gif_path="mario_agent.gif", mp4_path=None, num_episodes=10, fps=30):
    """
    Record the best episode of the agent playing Mario as a GIF (and optionally MP4).

    Args:
        model:        your trained ActorCritic model (already on device, eval mode)
        world:        Mario world number (e.g. 1)
        stage:        Mario stage number (e.g. 1)
        gif_path:     output path for the GIF
        mp4_path:     optional output path for an MP4 (requires imageio[ffmpeg])
        num_episodes: how many episodes to try; saves the best (flag > reward)
        fps:          frames per second for the output (30 recommended)
    """
    best_frames = []
    best_reward = -float("inf")
    best_info = None
    flag_count = 0

    for episode in range(num_episodes):
        # Single env — use raw RGB render, skip custom wrappers for display
        env = gym_super_mario_bros.make(f"SuperMarioBros-{world}-{stage}-v0")
        env = JoypadSpace(env, SIMPLE_MOVEMENT)

        # Wrapped env for the model's observations
        wrapped = make_env(world, stage)

        result = env.reset()
        raw_obs = result[0] if isinstance(result, tuple) else result
        wrapped.reset()

        done = False
        episode_reward = 0
        frames = [raw_obs.copy()]

        # We need to keep wrapped and raw in sync step-by-step.
        # Because CustomSkipFrame internally steps 4x, we mirror that in raw_env too.
        # The cleanest approach: drive off wrapped's done signal, capture raw frames at each sub-step.

        # Reset both cleanly
        raw_obs = env.reset()
        if isinstance(raw_obs, tuple):
            raw_obs = raw_obs[0]
        wrapped_state = wrapped.reset()

        frames = [raw_obs.copy()]

        while not done:
            state_tensor = torch.FloatTensor(np.array(wrapped_state)).unsqueeze(0).to(device)
            with torch.no_grad():
                action, _ = model.get_action(state_tensor)

            # Step the wrapped env (handles 4-frame skip internally)
            wrapped_state, reward, done, info = wrapped.step(action)
            episode_reward += reward

            # Mirror the same 4 sub-steps on the raw display env to stay in sync
            for _ in range(4):
                result = env.step(action)
                if len(result) == 5:
                    raw_obs, _, t, tr, _ = result
                    raw_done = t or tr
                else:
                    raw_obs, _, raw_done, _ = result
                frames.append(raw_obs.copy())
                if raw_done:
                    break

        env.close()
        wrapped.close()

        flag = info.get("flag_get", False)
        if flag:
            flag_count += 1
            print(f"Episode {episode+1:>2} | Reward: {episode_reward:>7.2f} | FLAG GET ✅ | x_pos: {info['x_pos']}")
        else:
            print(f"Episode {episode+1:>2} | Reward: {episode_reward:>7.2f} | No flag   | x_pos: {info['x_pos']}")

        # Keep best: prefer flag, then highest reward
        is_better = (
            (flag and not (best_info and best_info.get("flag_get")))
            or (flag == best_info.get("flag_get", False) if best_info else True)
            and episode_reward > best_reward
        )
        if best_info is None or is_better:
            best_frames = frames
            best_reward = episode_reward
            best_info = info

    print(f"\nFlag get rate: {flag_count}/{num_episodes}")
    print(f"Best episode reward: {best_reward:.2f} | x_pos: {best_info['x_pos']}")
    print(f"Saving {len(best_frames)} frames...")

    # --- Save GIF ---
    pil_frames = [Image.fromarray(f.astype(np.uint8)) for f in best_frames]
    duration_ms = int(1000 / fps)
    pil_frames[0].save(
        gif_path,
        save_all=True,
        append_images=pil_frames[1:],
        duration=duration_ms,
        loop=0,
        optimize=False,   # keep optimize=False for smoother playback
    )
    print(f"✅ GIF saved → {gif_path}  ({len(best_frames)} frames @ {fps}fps)")

    # --- Save MP4 (optional, needs imageio[ffmpeg]) ---
    if mp4_path:
        writer = imageio.get_writer(mp4_path, fps=fps, codec="libx264", quality=8)
        for frame in best_frames:
            writer.append_data(frame.astype(np.uint8))
        writer.close()
        print(f"✅ MP4 saved → {mp4_path}")

    return gif_path


# --- Run it ---
record_agent(
    model,
    world=1,
    stage=1,
    gif_path="mario_1_1.gif",
    mp4_path="mario_1_1.mp4",   # remove this line if you don't want MP4
    num_episodes=10,
    fps=30,
)

/usr/local/lib/python3.12/dist-packages/gym/envs/registration.py:555: UserWarning: WARN: The environment SuperMarioBros-1-1-v0 is out of date. You should consider upgrading to version `v3`.
  logger.warn(
/usr/local/lib/python3.12/dist-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional information. Actual type: `<class 'numpy.ndarray'>`
  logger.warn(
/usr/local/lib/python3.12/dist-packages/gym/utils/passive_env_checker.py:219: DeprecationWarning: WARN: Core environment is written in old step API which returns one bool instead of two. It is recommended to rewrite the environment with new step API. 
  logger.deprecation(


Episode  1 | Reward:  167.50 | No flag   | x_pos: 1805
Episode  2 | Reward:  180.05 | No flag   | x_pos: 1950
Episode  3 | Reward:  231.95 | No flag   | x_pos: 2469
Episode  4 | Reward:  164.85 | No flag   | x_pos: 1785
Episode  5 | Reward:  309.65 | FLAG GET ✅ | x_pos: 3161
Episode  6 | Reward:  310.60 | FLAG GET ✅ | x_pos: 3161
Episode  7 | Reward:  233.15 | No flag   | x_pos: 2471
Episode  8 | Reward:  233.65 | No flag   | x_pos: 2472
Episode  9 | Reward:  165.80 | No flag   | x_pos: 1787
Episode 10 | Reward:  188.25 | No flag   | x_pos: 2017

Flag get rate: 2/10
Best episode reward: 310.60 | x_pos: 3161
Saving 1510 frames...
✅ GIF saved → mario_1_1.gif  (1510 frames @ 30fps)
✅ MP4 saved → mario_1_1.mp4


'mario_1_1.gif'